# Rivayat — ComfyUI on a Colab T4

The free image lane, moved off the 6 GB Quadro RTX 3000 and onto a Colab T4 (~15 GB).
The point is **not speed** — the T4 is the same Turing generation as the local card and is only
modestly faster. The point is **headroom**: 15 GB makes SDXL and FLUX.1-schnell possible, and
neither of those fits locally.

Full context, the local comparison numbers and the fallback path:
`tools/colab/README.md` in the Rivayat repo.

---

## ⚠️ Read these two warnings before you run anything

### 1. Google's Terms forbid this on the **free** tier

Colab's FAQ lists, as disallowed **from free-tier runtimes without a positive compute balance**:

> *"bypassing the notebook UI to interact primarily via a web UI"* and
> *"remote control such as SSH shells, remote desktops"*

Serving the ComfyUI API through a tunnel to an external client is exactly that. This is the same
policy Google used in 2023 to block `stable-diffusion-webui` on free Colab.

**So: run this on Colab Pro, Pro+, or pay-as-you-go compute units.** The restriction is scoped to
runtimes *without* a positive compute balance — buying the cheapest compute-unit bundle moves you
out of it. If you will not pay, use **Kaggle** (30 GPU-hours/week, T4×2 or P100) or a rented
GPU (RunPod, Vast.ai) instead. Do not run this on a free Colab account and then be surprised
when the runtime is terminated.

### 2. The tunnel is a public URL

`trycloudflare.com` URLs are unauthenticated and reachable by anyone who learns them. ComfyUI has
**no authentication of its own** — an exposed instance is an open image generator running on your
quota, and `/view` will serve anything in the output directory.

This notebook therefore never exposes ComfyUI directly. It puts a **token-gated reverse proxy** in
front of it, and tunnels the proxy. Every request that does not carry the shared secret gets a
401. Do not disable it, and do not paste the printed URL *and* token anywhere public.

---

## 1 · What did Colab actually give us?

Free/burst Colab does not promise a GPU at all, let alone a particular one. This cell reports what
was attached and **stops with a loud error if there is no GPU**, so you do not spend ten minutes
downloading 7 GB onto a CPU-only runtime.

It also prints the **compute capability**, which decides the model set:

- **7.5 (T4, Turing)** — fp16 yes, **bf16 no, fp8 no**. FLUX must be run as **GGUF**, not fp8.
- **8.0+ (A100/L4)** — bf16 fine; 8.9+ (L4) additionally does fp8.

In [ ]:
import subprocess, sys, shutil

print("=" * 68)
q = "name,memory.total,compute_cap,driver_version"
if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "\n*** NO GPU ATTACHED ***\n"
        "Runtime > Change runtime type > Hardware accelerator: GPU, then Runtime > Restart.\n"
        "Free Colab does not guarantee a GPU; if none is available, try again later."
    )

out = subprocess.run(
    ["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
if not out:
    raise SystemExit("\n*** nvidia-smi returned nothing — no GPU attached. ***")

name, mem, cc, drv = [f.strip() for f in out.split(",")]
vram_mib = int(mem.split()[0])
cc_f = float(cc)
print(f"GPU              : {name}")
print(f"VRAM             : {vram_mib} MiB ({vram_mib/1024:.1f} GiB)")
print(f"Compute capability: {cc}   driver {drv}")

import psutil, os
print(f"System RAM       : {psutil.virtual_memory().total/2**30:.1f} GiB")
print(f"Free disk (/)    : {psutil.disk_usage('/').free/2**30:.1f} GiB")
try:
    import torch
    print(f"torch            : {torch.__version__}  cuda={torch.version.cuda}")
except ImportError:
    print("torch            : not importable yet (installed in step 4)")
print("=" * 68)

# --- the two facts that drive the model-set choice ---
if vram_mib < 14000:
    print(f"\n!! WARNING: only {vram_mib} MiB of VRAM. SDXL at 1024^2 wants ~10-12 GiB and")
    print("   FLUX-schnell Q4 wants ~11 GiB. Use MODEL_SET='sd15' or expect OOM.")
if cc_f < 8.0:
    print("\nNOTE: compute capability < 8.0 (this is a T4/Turing).")
    print("   -> bf16 and fp8 are NOT supported. Do not use fp8 FLUX checkpoints here;")
    print("      MODEL_SET='flux-schnell' uses GGUF quantisation, which is fp16 maths.")
if vram_mib >= 14000:
    print("\nOK: >= 14 GiB VRAM. SDXL and FLUX-schnell GGUF are both in range.")

---

## 2 · Configuration

Everything you might want to change lives in this one cell.

**`MODEL_SET`** — what gets downloaded. Sizes are exact (verified against the Hugging Face CDN):

| set | contents | download | why |
| --- | --- | --- | --- |
| `sdxl` *(default)* | SDXL base 1.0 + LCM-LoRA-SDXL | **7.3 GB** | The default *because it genuinely uses the 15 GB*. Runs the three **existing** `tools/comfy-workflows/*.json` unchanged — same nodes, just a different `{{checkpoint}}` and 1024² instead of 512². |
| `flux-schnell` | FLUX.1-schnell Q4_K_S + T5-XXL Q5_K_M + CLIP-L + FLUX VAE | **10.8 GB** | The parts-decomposition experiment (research §3). Needs the two new `txt2img-flux-schnell-*.json` workflows. |
| `both` | all of the above | **18.1 GB** | Does **not** fit in a free 15 GB Google Drive. Runtime disk only. |
| `sd15` | dreamshaper_8 + LCM-LoRA-SD1.5 | **2.3 GB** | Byte-identical to the local lane, for A/B comparison. |

**`AUTH_TOKEN`** — leave blank and a 43-character secret is generated for you. Set it explicitly
only if you want the same token across sessions (e.g. it is already in your `.env`).

**Never commit the token.** It goes in `.env`, which is gitignored, not in `.env.example`.

In [ ]:
# @title Rivayat Colab lane — configuration { display-mode: "form" }

MODEL_SET = "sdxl"  # @param ["sdxl", "flux-schnell", "both", "sd15"]
USE_DRIVE = False  # @param {type:"boolean"}
AUTH_TOKEN = ""  # @param {type:"string"}
TUNNEL = "cloudflared"  # @param ["cloudflared", "none"]

# --- pins. Not `master`: master drifts and takes the determinism claim with it. ---
COMFYUI_COMMIT = "72865f4f27eaf5396f8f36370e0a2be3a9a090ee"  # tag v0.33.1, 2026-08-13
COMFYUI_GGUF_COMMIT = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"  # main @ 2026-01-12

# Linux has no Windows reserved-port problem, so the ComfyUI default 8188 is fine here.
# (Locally we use 8288 because 8188 sits in a WinNAT exclusion range — see
#  tools/comfy-workflows/README.md.) Only the PROXY port is ever tunnelled.
COMFY_PORT = 8188
PROXY_PORT = 8189

COMFY_DIR = "/content/ComfyUI"
DRIVE_CACHE = "/content/drive/MyDrive/rivayat-comfy-models"

import secrets, os
if not AUTH_TOKEN:
    AUTH_TOKEN = secrets.token_urlsafe(32)
    print("Generated a fresh shared secret for this session.")
if len(AUTH_TOKEN) < 16:
    raise SystemExit("AUTH_TOKEN must be at least 16 characters — the proxy refuses to start otherwise.")
os.environ["RV_AUTH_TOKEN"] = AUTH_TOKEN

print(f"MODEL_SET  = {MODEL_SET}")
print(f"USE_DRIVE  = {USE_DRIVE}")
print(f"TUNNEL     = {TUNNEL}")
print(f"AUTH_TOKEN = {AUTH_TOKEN[:4]}...{AUTH_TOKEN[-4:]}  ({len(AUTH_TOKEN)} chars, printed in full in step 8)")

---

## 3 · Optional — cache the weights in Google Drive

Colab's local disk is wiped on every disconnect. Without a cache, a reconnect re-downloads
7–11 GB. With `USE_DRIVE = True` the weights land in `MyDrive/rivayat-comfy-models/` and are
**symlinked** into ComfyUI's model folders, so a reconnect costs a mount instead of a download.

Three honest caveats:

1. **A free Google Drive is 15 GB, shared with Gmail and Photos.** `sdxl` (7.3 GB) fits;
   `flux-schnell` (10.8 GB) probably fits; `both` (18.1 GB) **does not**.
2. **Drive reads are slower than the Hugging Face CDN.** Loading a 6.9 GB checkpoint off Drive can
   take longer than re-downloading it. The cache saves your *bandwidth and quota*, not always your
   time. If HF is fast for you, leaving `USE_DRIVE = False` is a legitimate choice.
3. Mounting grants this notebook access to your whole Drive. It only ever writes under
   `rivayat-comfy-models/`, but the permission is broader than that — that is Colab's design, not
   this notebook's.

Skip the cell entirely if `USE_DRIVE` is `False`; it is a no-op.

In [ ]:
import os

MODEL_ROOT = "/content/models-cache"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODEL_ROOT = DRIVE_CACHE
    print(f"Weight cache -> {MODEL_ROOT}")
    print("Reminder: a free Drive is 15 GB total. MODEL_SET='both' (18.1 GB) will not fit.")
else:
    print(f"Weight cache -> {MODEL_ROOT} (ephemeral: lost on disconnect)")

os.makedirs(MODEL_ROOT, exist_ok=True)
print("ready")

---

## 4 · Install ComfyUI, pinned

Cloned then `checkout`ed to an exact commit — **`v0.33.1` = `72865f4f…`**, the release that matches
the locally benchmarked `0.33.0`. Tracking `master` would silently change sampler numerics between
sessions, and the repo's determinism rule (`CLAUDE.md` §1) says the same spec must produce the same
bytes.

`requirements.txt` lists `torch` unpinned, so pip sees Colab's preinstalled CUDA build as already
satisfying it and leaves it alone. The cell prints the torch version before and after so you can
confirm nothing was swapped underneath you.

**Custom nodes stay off** (`--disable-all-custom-nodes`, applied at launch in step 8). The single
exception is `ComfyUI-GGUF`, needed for the FLUX GGUF loaders, and only when `MODEL_SET` calls for
it — pinned to a commit and explicitly whitelisted, exactly as
`tools/comfy-workflows/README.md` §7.6 requires.

In [ ]:
import subprocess, sys, os, textwrap

def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = r.stdout.strip().splitlines()[-12:]
    print(textwrap.indent("\n".join(tail), "  "))
    if r.returncode != 0:
        raise SystemExit(f"command failed ({r.returncode}): {cmd}")

try:
    import torch
    print(f"torch BEFORE: {torch.__version__} (cuda {torch.version.cuda})")
except ImportError:
    print("torch BEFORE: absent")

if not os.path.isdir(COMFY_DIR):
    sh(f"git clone --filter=blob:none https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}")
sh(f"git -C {COMFY_DIR} fetch --depth 1 origin {COMFYUI_COMMIT}")
sh(f"git -C {COMFY_DIR} checkout --detach {COMFYUI_COMMIT}")
head = subprocess.run(f"git -C {COMFY_DIR} rev-parse HEAD", shell=True,
                      capture_output=True, text=True).stdout.strip()
assert head == COMFYUI_COMMIT, f"pin failed: HEAD is {head}"
print(f"ComfyUI pinned at {head}")

sh(f"pip install -q -r {COMFY_DIR}/requirements.txt")

NEEDS_GGUF = MODEL_SET in ("flux-schnell", "both")
if NEEDS_GGUF:
    gguf_dir = f"{COMFY_DIR}/custom_nodes/ComfyUI-GGUF"
    if not os.path.isdir(gguf_dir):
        sh(f"git clone --filter=blob:none https://github.com/city96/ComfyUI-GGUF {gguf_dir}")
    sh(f"git -C {gguf_dir} fetch --depth 1 origin {COMFYUI_GGUF_COMMIT}")
    sh(f"git -C {gguf_dir} checkout --detach {COMFYUI_GGUF_COMMIT}")
    sh(f"pip install -q gguf>=0.13.0 numpy")
    print("\n!! UNVERIFIED: ComfyUI-GGUF's last commit is 2026-01-12; ComfyUI v0.33.1 is 2026-08-13.")
    print("   That pin has not been tested against this ComfyUI release. If the GGUF loader")
    print("   nodes fail to register, bump COMFYUI_GGUF_COMMIT to the current main and re-run.")

import importlib
try:
    importlib.reload(torch)
except Exception:
    import torch
print(f"\ntorch AFTER : {torch.__version__} (cuda {torch.version.cuda})")

---

## 5 · Download the model set

Every entry carries the **exact byte size and SHA-256** read from the Hugging Face CDN on
2026-08-23. The cell verifies both after download and refuses a corrupt or truncated file, so a
half-finished download cannot quietly become a "model that produces noise".

Two sourcing notes worth knowing:

- **`black-forest-labs/FLUX.1-schnell` is gated.** Fetching `ae.safetensors` from it anonymously
  returns **HTTP 401** — it needs an accepted licence and an HF token, which would break an
  unattended notebook. This cell instead pulls the identical VAE from
  `Comfy-Org/Lumina_Image_2.0_Repackaged`, which is ungated and ships the same 335,304,388-byte
  FLUX autoencoder. *Its SHA-256 is recorded below; equality with the gated original is inferred
  from the byte size, not proven.*
- **FLUX is GGUF, not fp8.** A T4 is compute capability 7.5 and has no fp8 path at all. GGUF
  dequantises to fp16, which Turing does natively.

Files go to `MODEL_ROOT` and are symlinked into ComfyUI, so switching `USE_DRIVE` does not move
anything twice.

In [ ]:
import hashlib, os, subprocess, textwrap

# name -> (comfy model subdir, url, size_bytes, sha256)   [verified 2026-08-23 via HF CDN headers]
CATALOG = {
    # --- sdxl ---
    "sd_xl_base_1.0.safetensors": (
        "checkpoints",
        "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
        6938078334, "31e35c80fc4829d14f90153f4c74cd59c90b779f6afe05a74cd6120b893f7e5b"),
    "lcm-lora-sdxl.safetensors": (
        "loras",
        "https://huggingface.co/latent-consistency/lcm-lora-sdxl/resolve/main/pytorch_lora_weights.safetensors",
        393855224, "a764e6859b6e04047cd761c08ff0cee96413a8e004c9f07707530cd776b19141"),
    # --- flux-schnell (GGUF: T4 has no fp8) ---
    "flux1-schnell-Q4_K_S.gguf": (
        "unet",
        "https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q4_K_S.gguf",
        6783943712, "4fd16477b3a5296d0cf722c4b92a9fd7f30d09ac7495826e4465d8de9c9fd973"),
    "t5-v1_1-xxl-encoder-Q5_K_M.gguf": (
        "text_encoders",
        "https://huggingface.co/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/t5-v1_1-xxl-encoder-Q5_K_M.gguf",
        3386856640, "b51cbb10b1a7aac6dd1c3b62f0ed908bfd06e0b42d2f3577d43e061361f51dae"),
    "clip_l.safetensors": (
        "text_encoders",
        "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
        246144152, "660c6f5b1abae9dc498ac2d21e1347d2abdb0cf6c0c0c8576cd796491d9a6cdd"),
    "ae.safetensors": (  # ungated mirror of the FLUX VAE; BFL's own repo 401s anonymously
        "vae",
        "https://huggingface.co/Comfy-Org/Lumina_Image_2.0_Repackaged/resolve/main/split_files/vae/ae.safetensors",
        335304388, "afc8e28272cd15db3919bacdb6918ce9c1ed22e96cb12c4d5ed0fba823529e38"),
    # --- sd15 parity set ---
    "dreamshaper_8.safetensors": (
        "checkpoints",
        "https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors",
        2132625894, "879db523c30d3b9017143d56705015e15a2cb5628762c11d086fed9538abd7fd"),
    "lcm-lora-sdv1-5.safetensors": (
        "loras",
        "https://huggingface.co/latent-consistency/lcm-lora-sdv1-5/resolve/main/pytorch_lora_weights.safetensors",
        134621556, "8f90d840e075ff588a58e22c6586e2ae9a6f7922996ee6649a7f01072333afe4"),
}

SETS = {
    "sdxl":         ["sd_xl_base_1.0.safetensors", "lcm-lora-sdxl.safetensors"],
    "flux-schnell": ["flux1-schnell-Q4_K_S.gguf", "t5-v1_1-xxl-encoder-Q5_K_M.gguf",
                     "clip_l.safetensors", "ae.safetensors"],
    "sd15":         ["dreamshaper_8.safetensors", "lcm-lora-sdv1-5.safetensors"],
}
SETS["both"] = SETS["sdxl"] + SETS["flux-schnell"]

wanted = SETS[MODEL_SET]
total = sum(CATALOG[n][2] for n in wanted)
print(f"MODEL_SET={MODEL_SET}: {len(wanted)} files, {total/2**30:.2f} GiB\n")


def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


for name in wanted:
    subdir, url, size, sha = CATALOG[name]
    cached = os.path.join(MODEL_ROOT, name)
    target_dir = os.path.join(COMFY_DIR, "models", subdir)
    os.makedirs(target_dir, exist_ok=True)
    target = os.path.join(target_dir, name)

    if os.path.exists(cached) and os.path.getsize(cached) == size:
        print(f"[cached] {name}")
    else:
        print(f"[fetch ] {name}  ({size/2**30:.2f} GiB)")
        # -c resumes a partial file; wget is already on the Colab image.
        rc = subprocess.run(["wget", "-q", "--show-progress", "--progress=bar:force:noscroll",
                             "-c", "-O", cached, url]).returncode
        if rc != 0:
            raise SystemExit(f"download failed for {name}")

    got = os.path.getsize(cached)
    if got != size:
        os.remove(cached)
        raise SystemExit(f"{name}: size {got} != expected {size} (deleted; re-run this cell)")
    digest = sha256_of(cached)
    if digest != sha:
        os.remove(cached)
        raise SystemExit(f"{name}: sha256 {digest}\n  expected {sha}\n  (deleted; re-run this cell)")
    print(f"         sha256 ok  {digest[:16]}...")

    if os.path.islink(target) or os.path.exists(target):
        os.remove(target)
    os.symlink(cached, target)
    print(f"         -> models/{subdir}/{name}")

print("\nAll weights present and verified.")

---

## 6 · Write the token gate

**This is the security boundary.** ComfyUI has no authentication whatsoever, so the tunnel must not
point at it. The chain is:

```
cloudflared  →  rv_auth_proxy (:8189)  →  ComfyUI (127.0.0.1:8188)
```

The proxy is the only thing the public URL can reach. It accepts the shared secret three ways —
`Authorization: Bearer …` (what the Rivayat `ComfyUiAdapter` sends), `X-Rivayat-Token: …`, or
`?rv_token=…` (which also sets a cookie, so you can open the ComfyUI web UI in a browser through
the same tunnel). Everything else gets a 401 with a body that reveals nothing about the runtime.
Comparison is `hmac.compare_digest`, so the token cannot be guessed byte by byte.

`/rv-health` is deliberately **unauthenticated** and returns only `{"ok": true}` — the keep-alive
cell polls it without leaking queue or model state.

*This exact file was run against a real ComfyUI 0.33.0 and passed 18 assertions: the gate, header
and query-string auth, transparent pass-through of `/system_stats` and the 1.6 MB `/object_info`,
and a full `POST /prompt` → `/history` → `/view` PNG round trip. See `tools/colab/README.md`.*

In [ ]:
%%writefile /content/rv_auth_proxy.py
#!/usr/bin/env python3
"""Token-gated reverse proxy in front of ComfyUI.

ComfyUI has no authentication of any kind. Exposing it through a public tunnel
without a gate makes it an open image generator on the internet, on someone
else's quota. This proxy is the gate: it is the only thing the tunnel can see,
and it refuses every request that does not present the shared secret.

Chain:  cloudflared  ->  this proxy (:PORT)  ->  ComfyUI (127.0.0.1:UPSTREAM)

The secret may arrive three ways, in this order of preference:
  1. Authorization: Bearer <token>     <- what the Rivayat adapter sends
  2. X-Rivayat-Token: <token>
  3. ?rv_token=<token> in the query string, which also sets a session cookie
     so a human can browse the ComfyUI web UI through the tunnel (browsers
     cannot attach custom headers to sub-resource or WebSocket requests).

Everything except /rv-health requires the token. /rv-health is deliberately
unauthenticated and deliberately says nothing but {"ok": true} plus a nonce-free
timestamp-free constant, so the keep-alive cell can poll it without leaking
whether a model is loaded or what the queue contains.
"""

from __future__ import annotations

import argparse
import asyncio
import hmac
import sys

import aiohttp
from aiohttp import web

# Headers that describe this hop only and must not be forwarded verbatim.
HOP_BY_HOP = frozenset(
    {
        "connection",
        "keep-alive",
        "proxy-authenticate",
        "proxy-authorization",
        "te",
        "trailers",
        "transfer-encoding",
        "upgrade",
        "host",
        "content-length",
        "content-encoding",
    }
)

# Our own auth material never reaches ComfyUI.
STRIP_UPSTREAM = frozenset({"authorization", "x-rivayat-token", "cookie"})

COOKIE_NAME = "rv_token"
HEALTH_PATH = "/rv-health"


def presented_token(request: web.Request) -> str:
    auth = request.headers.get("Authorization", "")
    if auth.startswith("Bearer "):
        return auth[7:].strip()
    header = request.headers.get("X-Rivayat-Token")
    if header:
        return header.strip()
    query = request.query.get("rv_token")
    if query:
        return query.strip()
    return request.cookies.get(COOKIE_NAME, "")


def authorised(request: web.Request, secret: str) -> bool:
    # compare_digest, not ==, so the tunnel cannot be probed byte by byte.
    return hmac.compare_digest(presented_token(request), secret)


def make_app(secret: str, upstream: str) -> web.Application:
    app = web.Application(client_max_size=1024 * 1024 * 256)

    async def health(_request: web.Request) -> web.Response:
        return web.json_response({"ok": True})

    async def proxy(request: web.Request) -> web.StreamResponse:
        if not authorised(request, secret):
            return web.json_response(
                {"error": "unauthorised", "hint": "send Authorization: Bearer <COMFYUI_AUTH_TOKEN>"},
                status=401,
            )

        target = f"{upstream}{request.rel_url}"
        session: aiohttp.ClientSession = request.app["session"]

        # --- WebSocket upgrade (ComfyUI /ws progress channel) ---
        if request.headers.get("Upgrade", "").lower() == "websocket":
            client_ws = web.WebSocketResponse()
            await client_ws.prepare(request)
            ws_target = target.replace("http://", "ws://", 1).replace("https://", "wss://", 1)
            async with session.ws_connect(ws_target) as upstream_ws:

                async def pump(src, dst) -> None:
                    async for msg in src:
                        if msg.type == aiohttp.WSMsgType.TEXT:
                            await dst.send_str(msg.data)
                        elif msg.type == aiohttp.WSMsgType.BINARY:
                            await dst.send_bytes(msg.data)
                        elif msg.type in (aiohttp.WSMsgType.CLOSE, aiohttp.WSMsgType.CLOSING):
                            break

                await asyncio.wait(
                    [
                        asyncio.create_task(pump(client_ws, upstream_ws)),
                        asyncio.create_task(pump(upstream_ws, client_ws)),
                    ],
                    return_when=asyncio.FIRST_COMPLETED,
                )
            return client_ws

        # --- plain HTTP ---
        fwd = {k: v for k, v in request.headers.items() if k.lower() not in HOP_BY_HOP and k.lower() not in STRIP_UPSTREAM}
        body = await request.read()
        async with session.request(
            request.method, target, headers=fwd, data=body, allow_redirects=False
        ) as upstream_resp:
            out = web.StreamResponse(status=upstream_resp.status)
            for k, v in upstream_resp.headers.items():
                if k.lower() not in HOP_BY_HOP:
                    out.headers[k] = v
            # A ?rv_token= that just authenticated becomes a cookie, so the
            # ComfyUI web UI's own asset/WebSocket requests inherit the auth.
            if request.query.get("rv_token"):
                out.set_cookie(COOKIE_NAME, secret, httponly=True, samesite="Lax", path="/")
            await out.prepare(request)
            async for chunk in upstream_resp.content.iter_chunked(65536):
                await out.write(chunk)
            await out.write_eof()
            return out

    async def on_startup(a: web.Application) -> None:
        a["session"] = aiohttp.ClientSession(
            timeout=aiohttp.ClientTimeout(total=None, sock_connect=15, sock_read=None),
            auto_decompress=False,
        )

    async def on_cleanup(a: web.Application) -> None:
        await a["session"].close()

    app.on_startup.append(on_startup)
    app.on_cleanup.append(on_cleanup)
    app.router.add_get(HEALTH_PATH, health)
    app.router.add_route("*", "/{tail:.*}", proxy)
    return app


def main() -> int:
    ap = argparse.ArgumentParser(description="Token-gated reverse proxy for ComfyUI")
    ap.add_argument("--port", type=int, default=8289, help="port this proxy listens on")
    ap.add_argument("--upstream", default="http://127.0.0.1:8288", help="ComfyUI base URL")
    ap.add_argument("--token", required=True, help="shared secret clients must present")
    ap.add_argument("--bind", default="0.0.0.0")
    args = ap.parse_args()

    if len(args.token) < 16:
        print("refusing to start: token must be at least 16 characters", file=sys.stderr)
        return 2

    print(f"rv-auth-proxy: {args.bind}:{args.port} -> {args.upstream} (token gate ON)", flush=True)
    web.run_app(make_app(args.token, args.upstream), host=args.bind, port=args.port, print=None)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

---

## 7 · Install `cloudflared`

**Why cloudflared and not the alternatives:**

| option | account needed? | verdict |
| --- | --- | --- |
| **cloudflared quick tunnel** | **no** | Chosen. One command, random `*.trycloudflare.com` HTTPS URL, WebSockets supported (ComfyUI's `/ws` progress channel needs them). |
| `localtunnel` | no | Interstitial page that demands the tunnel's public IP before passing traffic — hostile to a machine client. |
| `ngrok` | **yes** — authtoken | Works, but adds a signup and a browser-warning interstitial on the free plan. |

Cloudflare documents quick tunnels as **for testing and demos, not production**, and caps them at
**200 concurrent requests**. For one adapter driving one ComfyUI, that is irrelevant. The URL is
random and changes every session — that is a rotation property, not a security control, which is
why step 6 exists.

In [ ]:
import subprocess, os, textwrap

if TUNNEL == "cloudflared":
    if not os.path.exists("/usr/local/bin/cloudflared"):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", url], check=True)
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    v = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True).stdout.strip()
    print(v)
else:
    print("TUNNEL='none' — ComfyUI will only be reachable from inside this runtime.")

---

## 8 · Launch, and print the one line you copy

Starts the three processes in order, waiting for each to be healthy:

1. **ComfyUI** on `127.0.0.1:8188`, bound to loopback so it is unreachable except through the proxy.
   Custom nodes are disabled; `ComfyUI-GGUF` is whitelisted only when the model set needs it.
2. **The token gate** on `0.0.0.0:8189`.
3. **cloudflared**, pointed at the gate — never at ComfyUI.

When it finishes it prints the exact `.env` lines for the Rivayat repo. Copy them into `.env`
(**not** `.env.example` — that file must never hold a secret).

In [ ]:
import subprocess, time, re, os, json, urllib.request, urllib.error, signal

LOGS = "/content/rv-logs"
os.makedirs(LOGS, exist_ok=True)
procs = globals().setdefault("RV_PROCS", {})


def kill(tag):
    p = procs.pop(tag, None)
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(10)
        except subprocess.TimeoutExpired:
            p.kill()


def spawn(tag, args, cwd=None):
    kill(tag)
    log = open(f"{LOGS}/{tag}.log", "wb")
    procs[tag] = subprocess.Popen(args, cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
    return procs[tag]


def wait_http(url, timeout, tag, headers=None):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=5) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        if procs.get(tag) and procs[tag].poll() is not None:
            print(open(f"{LOGS}/{tag}.log").read()[-3000:])
            raise SystemExit(f"{tag} exited early — log above")
        time.sleep(2)
    print(open(f"{LOGS}/{tag}.log").read()[-3000:])
    raise SystemExit(f"{tag} did not become healthy within {timeout}s — log above")


# --- 1. ComfyUI ------------------------------------------------------------
comfy_args = ["python", "main.py",
              "--listen", "127.0.0.1", "--port", str(COMFY_PORT),
              "--disable-auto-launch", "--preview-method", "none",
              "--disable-all-custom-nodes"]
if MODEL_SET in ("flux-schnell", "both"):
    comfy_args += ["--whitelist-custom-nodes", "ComfyUI-GGUF"]
print("starting ComfyUI:", " ".join(comfy_args))
spawn("comfyui", comfy_args, cwd=COMFY_DIR)
wait_http(f"http://127.0.0.1:{COMFY_PORT}/system_stats", 300, "comfyui")
stats = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats").read())
print(f"  ComfyUI {stats['system']['comfyui_version']} up on {stats['devices'][0]['name']}")

# --- 2. token gate ---------------------------------------------------------
print("starting rv-auth-proxy...")
spawn("proxy", ["python", "/content/rv_auth_proxy.py",
                "--port", str(PROXY_PORT),
                "--upstream", f"http://127.0.0.1:{COMFY_PORT}",
                "--token", AUTH_TOKEN])
wait_http(f"http://127.0.0.1:{PROXY_PORT}/rv-health", 60, "proxy")
# prove the gate is actually closed before anything is exposed
try:
    urllib.request.urlopen(f"http://127.0.0.1:{PROXY_PORT}/system_stats", timeout=10)
    raise SystemExit("REFUSING TO TUNNEL: the proxy served /system_stats without a token.")
except urllib.error.HTTPError as e:
    if e.code != 401:
        raise SystemExit(f"REFUSING TO TUNNEL: unauthenticated request returned {e.code}, expected 401.")
print("  gate verified: unauthenticated request -> 401")

# --- 3. tunnel -------------------------------------------------------------
PUBLIC_URL = f"http://127.0.0.1:{PROXY_PORT}"
if TUNNEL == "cloudflared":
    print("starting cloudflared quick tunnel...")
    spawn("cloudflared", ["cloudflared", "tunnel", "--no-autoupdate",
                          "--url", f"http://127.0.0.1:{PROXY_PORT}"])
    pat = re.compile(rb"https://[-a-z0-9]+\.trycloudflare\.com")
    deadline, found = time.time() + 120, None
    while time.time() < deadline and not found:
        time.sleep(2)
        m = pat.search(open(f"{LOGS}/cloudflared.log", "rb").read())
        if m:
            found = m.group(0).decode()
    if not found:
        print(open(f"{LOGS}/cloudflared.log").read()[-3000:])
        raise SystemExit("cloudflared did not produce a URL — log above")
    PUBLIC_URL = found
    wait_http(f"{PUBLIC_URL}/rv-health", 120, "cloudflared")
    print(f"  tunnel live: {PUBLIC_URL}")

globals()["PUBLIC_URL"] = PUBLIC_URL

print("\n" + "=" * 74)
print("  Paste these into  .env  in the Rivayat repo (NEVER into .env.example):")
print("=" * 74)
print()
print(f"COMFYUI_HOST={PUBLIC_URL}")
print(f"COMFYUI_AUTH_TOKEN={AUTH_TOKEN}")
print("RV_COMFYUI_REMOTE=true")
print()
print("=" * 74)
print("  This URL is public. The token is the only thing protecting it. Do not share both.")
print("  The URL dies with this session and is different every time. See step 11.")
print("=" * 74)

---

## 9 · Self-test through the public URL

Verifies the whole chain from the outside — the request leaves Colab, crosses Cloudflare's edge and
comes back — rather than just checking that a local port is open.

It asserts the gate is **closed** (401 without a token, 401 with a wrong one) before it asserts it
is **open** (200 with the right one), then confirms `/object_info` survives the tunnel intact. If
`MODEL_SET` includes FLUX it also checks that the GGUF loader nodes actually registered, which is
the one thing about the pinned custom node that could not be checked before running it.

In [ ]:
import urllib.request, urllib.error, json

fails = []


def call(path, token=None):
    h = {"Authorization": f"Bearer {token}"} if token else {}
    try:
        with urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + path, headers=h), timeout=60) as r:
            return r.status, r.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()


def check(name, got, want):
    ok = got == want
    fails.append(name) if not ok else None
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}: {got!r} (want {want!r})")


print(f"testing {PUBLIC_URL}\n")
check("unauthenticated /system_stats -> 401", call("/system_stats")[0], 401)
check("wrong token -> 401", call("/system_stats", "definitely-not-the-token")[0], 401)
check("correct token -> 200", call("/system_stats", AUTH_TOKEN)[0], 200)
check("/rv-health open -> 200", call("/rv-health")[0], 200)

st, body = call("/object_info", AUTH_TOKEN)
oi = json.loads(body) if st == 200 else {}
check("/object_info through the tunnel -> 200", st, 200)
print(f"         {len(body)} bytes, {len(oi)} node types")

if MODEL_SET in ("flux-schnell", "both"):
    for node in ("UnetLoaderGGUF", "DualCLIPLoaderGGUF"):
        check(f"{node} registered", node in oi, True)
    if "UnetLoaderGGUF" in oi:
        print("         GGUF files visible:",
              oi["UnetLoaderGGUF"]["input"]["required"]["unet_name"][0])
if MODEL_SET in ("sdxl", "both", "sd15"):
    print("         checkpoints visible:",
          oi["CheckpointLoaderSimple"]["input"]["required"]["ckpt_name"][0])
    print("         loras visible:",
          oi["LoraLoader"]["input"]["required"]["lora_name"][0])

print()
print("RESULT:", "ALL PASS — the lane is live" if not fails else f"{len(fails)} FAILURES: {fails}")

---

## 10 · Keep-alive and health

**Leave this cell running.** It does two jobs:

- **Keeps the runtime busy.** Colab reclaims runtimes that go idle (commonly cited at ~90 minutes,
  but Google explicitly does not publish the number and it varies). A cell that is actively
  executing and printing is not idle.
- **Tells you the moment the lane dies**, instead of you discovering it through a failed render.

It polls `/rv-health` through the public URL and prints one line a minute. It stops on its own
after `HOURS`, and it will **not** silently mask a failure: three consecutive failed polls end the
loop with a clear message.

This does not defeat Colab's **maximum session lifetime** (up to 12 h, less in practice, never
guaranteed) — nothing does. It only prevents the *idle* timeout.

In [ ]:
import time, urllib.request, urllib.error, datetime

HOURS = 6  # @param {type:"number"}

deadline = time.time() + HOURS * 3600
consecutive_failures = 0
print(f"polling {PUBLIC_URL}/rv-health every 60s until "
      f"{datetime.datetime.now() + datetime.timedelta(hours=HOURS):%H:%M}  (interrupt to stop)\n")

while time.time() < deadline:
    stamp = datetime.datetime.now().strftime("%H:%M:%S")
    err = "bad status"
    try:
        with urllib.request.urlopen(f"{PUBLIC_URL}/rv-health", timeout=20) as r:
            ok = r.status == 200
    except Exception as e:
        ok = False
        err = type(e).__name__
    if ok:
        consecutive_failures = 0
        try:
            import subprocess
            used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                                  capture_output=True, text=True).stdout.strip()
        except Exception:
            used = "?"
        print(f"{stamp}  healthy   vram={used}")
    else:
        consecutive_failures += 1
        print(f"{stamp}  UNHEALTHY ({err})  [{consecutive_failures}/3]")
        if consecutive_failures >= 3:
            print("\n*** The lane is down. See 'When the session drops' below. ***")
            break
    time.sleep(60)
else:
    print(f"\nReached the {HOURS}h limit. Re-run this cell to keep going, if the runtime survives.")

---

## 11 · When the session drops — and it will

Colab sessions end. Free/burst runtimes are capped at 12 hours and are frequently taken back much
sooner; the tunnel URL dies with them. Plan for it rather than fighting it.

**What survives what:**

| | tunnel URL | weights on runtime disk | weights in Drive | ComfyUI install |
| --- | --- | --- | --- | --- |
| Kernel restart (`Runtime > Restart`) | dead | **kept** | kept | **kept** |
| Runtime disconnect / reclaim | dead | **lost** | **kept** | lost |

**Recovery, in order:**

1. **Cell 8 alone** if the notebook is still connected but the tunnel died (cloudflared crashed, or
   the quick tunnel was cut). Re-running it kills the old processes and starts fresh.
2. **Cells 1–8** after a runtime restart where the disk survived. Steps 4 and 5 are idempotent —
   they see the pinned checkout and the verified weights and skip straight through.
3. **Everything, from cell 1** after a full disconnect. With `USE_DRIVE = True` this costs a Drive
   mount instead of a 7–11 GB download.

**Every recovery produces a new `COMFYUI_HOST`.** The URL is random per session — cell 8 prints the
new one and you update `.env` again. If you set `AUTH_TOKEN` explicitly in cell 2, the token at
least stays stable and only the host line changes.

**When Colab will not give you a GPU at all**, or you would rather not keep re-pasting a URL:
switch back to the local lane — `COMFYUI_HOST=http://127.0.0.1:8288`, `RV_COMFYUI_REMOTE=false`,
and run `tools/scripts/comfy-start.ps1`. You lose SDXL and FLUX and go back to SD 1.5 at 768²,
but it is always there and it is genuinely free.

---

## 12 · Shut it down

Closes the tunnel first, so the public URL stops answering before anything behind it is torn down.
Run this when you are finished, or before re-running cell 8 by hand.

Closing the browser tab does *not* do this promptly — the runtime lingers, and so does the tunnel.

In [ ]:
for tag in ("cloudflared", "proxy", "comfyui"):   # tunnel first
    p = globals().get("RV_PROCS", {}).pop(tag, None)
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(10)
        except Exception:
            p.kill()
        print(f"stopped {tag}")
    else:
        print(f"{tag} was not running")
print("\nThe public URL is dead. COMFYUI_HOST in .env now points at nothing —")
print("fall back to the local lane (http://127.0.0.1:8288, RV_COMFYUI_REMOTE=false)")
print("or re-run cells 4-8 to get a new URL.")